# iFood Catalog Intelligence: QLoRA Fine-Tuning of Qwen2.5-1.5B-Instruct

**Portfólio de Engenharia de IA — Projeto 2 (`ifood-catalog-slm-qlora`)**
- **Dataset (100% Real EAN-13 SKUs):** [`LucasFig0/brazilian-delivery-catalog-sft`](https://huggingface.co/datasets/LucasFig0/brazilian-delivery-catalog-sft) (800 train / 150 validation / 200 test)
- **Modelo Base:** `Qwen/Qwen2.5-1.5B-Instruct` (Quantização 4-bit NormalFloat via `bitsandbytes`)
- **Arquitetura PEFT:** LoRA (`r=16`, `lora_alpha=32`, `target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']`) com *Completion-Only Loss Masking*
- **Adaptador Publicado:** [`LucasFig0/qwen2.5-1.5b-ifood-catalog-qlora`](https://huggingface.co/LucasFig0/qwen2.5-1.5b-ifood-catalog-qlora)

In [ ]:
# [Célula 2] Instalação enxuta (sem conflitar com pyarrow/cudf nativos do Kaggle)
print('[1/7] Instalando bitsandbytes e peft...', flush=True)
!pip install -q -U bitsandbytes peft
print('[1/7] Instalação concluída com sucesso!', flush=True)

In [ ]:
# [Célula 3] Trava de GPU única (evita conflito DataParallel em T4 x2) e Imports
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import json
import math
import re
import time
from pathlib import Path
from typing import Any, Literal, Optional

import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

assert torch.cuda.is_available(), 'ERRO: Ative o acelerador GPU T4 nas configurações do Kaggle!'
print(f'[2/7] GPU Ativa: {torch.cuda.get_device_name(0)} | Dispositivos visíveis: {torch.cuda.device_count()} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB', flush=True)

BASE_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
HF_DATASET_ID = 'LucasFig0/brazilian-delivery-catalog-sft'
HF_ADAPTER_REPO_ID = 'LucasFig0/qwen2.5-1.5b-ifood-catalog-qlora'

hf_token = os.environ.get('HF_TOKEN', '').strip()
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret('HF_TOKEN').strip()
        os.environ['HF_TOKEN'] = hf_token
        print('[2/7] HF_TOKEN carregado via Kaggle Secrets (modo não-interativo).', flush=True)
    except Exception as e:
        print(f'[Aviso] Kaggle Secret HF_TOKEN não anexado ({e}). O treino rodará normalmente.', flush=True)

In [ ]:
# [Célula 4] Contrato Pydantic v2, Guardrail de Segurança Alimentar e Avaliador
CatalogCategory = Literal[
    'Lanches', 'Pizzas', 'Brasileira', 'Japonesa', 'Açaí e Sorvetes',
    'Bebidas', 'Sobremesas', 'Saudável e Fit', 'Padaria e Cafeteria', 'Mercado'
]
MainProteinOrBase = Literal[
    'Bovino', 'Frango', 'Suíno', 'Peixe e Frutos do Mar',
    'Vegetariano/Vegano', 'Massa/Queijo', 'Açaí/Frutas', 'Bebida/Outro'
]
PortionSize = Literal[
    'Individual', 'Casal (2 Pessoas)', 'Família (3+ Pessoas)',
    'Porção/Acompanhamento', 'Unitário'
]
DietaryTag = Literal[
    'Zero Açúcar', 'Sem Lactose', 'Sem Glúten',
    'Vegano', 'Vegetariano', 'Low Carb/Fit'
]
ANIMAL_PROTEINS = {'Bovino', 'Frango', 'Suíno', 'Peixe e Frutos do Mar'}
DIETARY_TAGS_VOCAB = ('Zero Açúcar', 'Sem Lactose', 'Sem Glúten', 'Vegano', 'Vegetariano', 'Low Carb/Fit')
SCALAR_FIELDS = ('category', 'main_protein_or_base', 'portion_size', 'is_combo', 'volume_or_weight')

class EnrichedCatalogItem(BaseModel):
    model_config = ConfigDict(extra='forbid')
    normalized_title: str = Field(..., min_length=3, max_length=120)
    category: CatalogCategory
    main_protein_or_base: MainProteinOrBase
    portion_size: PortionSize
    dietary_tags: list[DietaryTag] = Field(default_factory=list)
    is_combo: bool
    volume_or_weight: Optional[str] = Field(default=None)
    components: list[str] = Field(..., min_length=1, max_length=12)
    search_synonyms: list[str] = Field(..., min_length=2, max_length=6)

    @field_validator('volume_or_weight')
    @classmethod
    def normalize_measure(cls, value: Optional[str]) -> Optional[str]:
        if value is None:
            return None
        cleaned = value.strip()
        if not cleaned or cleaned.lower() in {'null', 'none', 'n/a', '-'}:
            return None
        cleaned = cleaned.replace(',', '.')
        match = re.fullmatch(r'(\d+(?:\.\d+)?)\s*(ml|l|g|kg|un)', cleaned, flags=re.IGNORECASE)
        if not match:
            raise ValueError(f'Medida inválida: {value}')
        amount, unit = match.group(1), match.group(2).lower()
        return f'{amount}{"L" if unit == "l" else unit}'

    @model_validator(mode='after')
    def validate_food_safety(self) -> 'EnrichedCatalogItem':
        if self.main_protein_or_base in ANIMAL_PROTEINS:
            forbidden = {'Vegano', 'Vegetariano'}.intersection(set(self.dietary_tags))
            if forbidden:
                raise ValueError(f'Violação de segurança alimentar: {forbidden}')
        return self

def parse_slm_json_output(raw_text: str) -> tuple[Optional[EnrichedCatalogItem], Optional[str], bool]:
    if not raw_text or not raw_text.strip():
        return None, 'Empty output', False
    text = raw_text.strip()
    repaired = False
    fence = chr(96) * 3
    if fence in text:
        repaired = True
        text = re.sub(rf'^{fence}(?:json)?\s*', '', text, flags=re.IGNORECASE)
        text = re.sub(rf'\s*{fence}$', '', text).strip()
    if not (text.startswith('{') and text.endswith('}')):
        start, end = text.find('{'), text.rfind('}')
        if start != -1 and end != -1 and end > start:
            repaired = True
            text = text[start : end + 1]
        else:
            return None, 'No JSON boundaries', repaired
    try:
        payload = json.loads(text)
        if not isinstance(payload, dict):
            return None, 'Root not dict', repaired
        return EnrichedCatalogItem.model_validate(payload), None, repaired
    except Exception as exc:
        return None, str(exc), repaired

def _tokenize_norm(items: list[str]) -> set[str]:
    return {w for text in items for w in re.findall(r'[a-zA-ZÀ-ÿ0-9]+', text.lower()) if len(w) >= 3}

def compute_token_overlap_f1(pred_list: list[str], gt_list: list[str]) -> float:
    p, g = _tokenize_norm(pred_list), _tokenize_norm(gt_list)
    if not p and not g: return 1.0
    if not p or not g: return 0.0
    inter = len(p.intersection(g))
    prec, rec = inter / len(p), inter / len(g)
    return (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

def evaluate_predictions(raw_outputs: list[str], ground_truths: list[dict], latencies_ms: list[float]) -> dict[str, Any]:
    total = len(ground_truths)
    valid_schema, strict_clean, safety_violations = 0, 0, 0
    field_hits = {f: 0 for f in SCALAR_FIELDS}
    comp_f1_sum, syn_f1_sum = 0.0, 0.0
    tp = {t: 0 for t in DIETARY_TAGS_VOCAB}
    fp = {t: 0 for t in DIETARY_TAGS_VOCAB}
    fn = {t: 0 for t in DIETARY_TAGS_VOCAB}

    for raw_text, gt_dict in zip(raw_outputs, ground_truths):
        gt = EnrichedCatalogItem.model_validate(gt_dict)
        pred, err, repaired = parse_slm_json_output(raw_text)
        if gt.main_protein_or_base in ANIMAL_PROTEINS and ('vegano' in raw_text.lower() or 'vegetariano' in raw_text.lower()):
            if '"dietary_tags"' in raw_text.lower():
                m = re.search(r'"dietary_tags"\s*:\s*\[([^\]]*)\]', raw_text.lower())
                if m and ('vegano' in m.group(1) or 'vegetariano' in m.group(1)):
                    safety_violations += 1
        gt_tags = set(gt.dietary_tags)
        if pred is None:
            for t in gt_tags: fn[t] += 1
            continue
        valid_schema += 1
        if not repaired: strict_clean += 1
        for f in SCALAR_FIELDS:
            if getattr(pred, f) == getattr(gt, f): field_hits[f] += 1
        comp_f1_sum += compute_token_overlap_f1(pred.components, gt.components)
        syn_f1_sum += compute_token_overlap_f1(pred.search_synonyms, gt.search_synonyms)
        pred_tags = set(pred.dietary_tags)
        for t in DIETARY_TAGS_VOCAB:
            if t in pred_tags and t in gt_tags: tp[t] += 1
            elif t in pred_tags and t not in gt_tags: fp[t] += 1
            elif t not in pred_tags and t in gt_tags: fn[t] += 1

    tot_tp, tot_fp, tot_fn = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro_f1 = (2 * tot_tp / (2 * tot_tp + tot_fp + tot_fn)) if (2 * tot_tp + tot_fp + tot_fn) > 0 else 1.0
    per_tag_f1 = {t: round((2*tp[t]/(2*tp[t]+fp[t]+fn[t])) if (2*tp[t]+fp[t]+fn[t])>0 else 1.0, 4) for t in DIETARY_TAGS_VOCAB}
    macro_f1 = sum(per_tag_f1.values()) / len(DIETARY_TAGS_VOCAB)
    field_acc = {f: round(field_hits[f] / total, 4) for f in SCALAR_FIELDS}
    s_lat = sorted(latencies_ms)
    return {
        'total_samples': total,
        'json_schema_compliance_rate': round(valid_schema / total, 4),
        'strict_clean_json_rate': round(strict_clean / total, 4),
        'mean_field_exact_match': round(sum(field_acc.values()) / len(SCALAR_FIELDS), 4),
        'field_accuracies': field_acc,
        'dietary_tags_micro_f1': round(micro_f1, 4),
        'dietary_tags_macro_f1': round(macro_f1, 4),
        'components_token_f1': round(comp_f1_sum / total, 4),
        'search_synonyms_token_f1': round(syn_f1_sum / total, 4),
        'food_safety_violations': safety_violations,
        'food_safety_violation_rate': round(safety_violations / total, 4),
        'latency': {
            'p50_ms': round(s_lat[len(s_lat)//2], 2),
            'p95_ms': round(s_lat[int(len(s_lat)*0.95)], 2),
            'mean_ms': round(sum(s_lat)/len(s_lat), 2),
        },
    }
print('[3/7] Schemas Pydantic v2 e motor de avaliação carregados!', flush=True)

In [ ]:
# [Célula 5] Carregamento do Dataset Real (1.150 SKUs com código EAN-13) do Hugging Face Hub
print(f'[4/7] Baixando dataset {HF_DATASET_ID}...', flush=True)
dataset = load_dataset(HF_DATASET_ID, token=hf_token if hf_token else None)
print(dataset, flush=True)
print('Exemplo Real de Treino (SKU_ID):', dataset['train'][0]['sku_id'], flush=True)

In [ ]:
# [Célula 6] Carregamento do Qwen2.5-1.5B-Instruct em 4-bit NF4 na GPU 0 e Benchmark Zero-Shot
print(f'[5/7] Carregando {BASE_MODEL_ID} em 4-bit NF4 na GPU 0...', flush=True)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 0},
    torch_dtype=torch.float16,
)
base_model.eval()
print(f'Modelo carregado! VRAM Alocada: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB', flush=True)

def run_batched_inference(model, tok, test_split, batch_size: int = 16) -> tuple[list[str], list[float]]:
    tok.padding_side = 'left'
    prompts = [
        tok.apply_chat_template(row['messages'][:2], tokenize=False, add_generation_prompt=True)
        for row in test_split
    ]
    outputs_text: list[str] = []
    latencies_ms: list[float] = []

    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i : i + batch_size]
        inputs = tok(batch_prompts, return_tensors='pt', padding=True, truncation=True, max_length=640).to('cuda:0')
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=260,
                do_sample=False,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )
        torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        per_item_ms = elapsed_ms / len(batch_prompts)
        input_len = inputs['input_ids'].shape[1]
        for seq in gen_ids:
            decoded = tok.decode(seq[input_len:], skip_special_tokens=True)
            outputs_text.append(decoded)
            latencies_ms.append(per_item_ms)
        print(f'  -> Inferência concluída: {min(i + batch_size, len(prompts))}/{len(prompts)} SKUs ({per_item_ms:.1f} ms/SKU)', flush=True)
    return outputs_text, latencies_ms

print('[5/7] Avaliando Modelo Base Zero-Shot nos 200 SKUs reais de Holdout...', flush=True)
test_rows = list(dataset['test'])
gt_list = [r['ground_truth'] for r in test_rows]
base_outputs, base_latencies = run_batched_inference(base_model, tokenizer, test_rows, batch_size=16)
base_metrics = evaluate_predictions(base_outputs, gt_list, base_latencies)
print('\n=== Métricas Modelo Base (Zero-Shot) ===', flush=True)
print(json.dumps(base_metrics, indent=2, ensure_ascii=False), flush=True)

In [ ]:
# [Célula 7] Preparação dos Tensores com Completion-Only Loss Masking e Treinamento QLoRA
print('[6/7] Preparando tensores e iniciando Fine-Tuning QLoRA...', flush=True)
tokenizer.padding_side = 'right'

def tokenize_with_completion_mask(example: dict[str, Any], max_length: int = 768) -> dict[str, Any]:
    prompt_text = tokenizer.apply_chat_template(example['messages'][:2], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=max_length)['input_ids']
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    labels = labels[:len(full_ids)]
    return {
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
    }

train_tokenized = dataset['train'].map(tokenize_with_completion_mask, remove_columns=dataset['train'].column_names)
val_tokenized = dataset['validation'].map(tokenize_with_completion_mask, remove_columns=dataset['validation'].column_names)

peft_model = prepare_model_for_kbit_training(base_model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
peft_model = get_peft_model(peft_model, lora_config)
peft_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir='/kaggle/working/lora_checkpoints',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    num_train_epochs=2,
    optim='paged_adamw_8bit',
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=25,
    save_strategy='no',
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors='pt', padding=True),
)

t_train_start = time.perf_counter()
train_result = trainer.train()
train_duration_min = round((time.perf_counter() - t_train_start) / 60.0, 2)
print(f'\n[6/7] Treinamento QLoRA concluído em {train_duration_min} minutos! Loss final: {train_result.training_loss:.4f}', flush=True)

In [ ]:
# [Célula 8] Avaliação Pós-SFT (QLoRA) nos mesmos 200 SKUs Reais de Holdout
print('[7/7] Avaliando Modelo Fine-Tuned QLoRA nos 200 SKUs de Holdout...', flush=True)
peft_model.eval()
peft_model.config.use_cache = True
ft_outputs, ft_latencies = run_batched_inference(peft_model, tokenizer, test_rows, batch_size=16)
ft_metrics = evaluate_predictions(ft_outputs, gt_list, ft_latencies)

comparison_df = pd.DataFrame([
    {
        'Métrica': 'JSON Schema Compliance Rate (%)',
        'Base Zero-Shot': round(base_metrics['json_schema_compliance_rate'] * 100, 2),
        'Fine-Tuned QLoRA': round(ft_metrics['json_schema_compliance_rate'] * 100, 2),
    },
    {
        'Métrica': 'Strict Clean JSON (Sem Reparo) (%)',
        'Base Zero-Shot': round(base_metrics['strict_clean_json_rate'] * 100, 2),
        'Fine-Tuned QLoRA': round(ft_metrics['strict_clean_json_rate'] * 100, 2),
    },
    {
        'Métrica': 'Mean Field Exact Match (%)',
        'Base Zero-Shot': round(base_metrics['mean_field_exact_match'] * 100, 2),
        'Fine-Tuned QLoRA': round(ft_metrics['mean_field_exact_match'] * 100, 2),
    },
    {
        'Métrica': 'Category Accuracy (%)',
        'Base Zero-Shot': round(base_metrics['field_accuracies']['category'] * 100, 2),
        'Fine-Tuned QLoRA': round(ft_metrics['field_accuracies']['category'] * 100, 2),
    },
    {
        'Métrica': 'Main Protein/Base Accuracy (%)',
        'Base Zero-Shot': round(base_metrics['field_accuracies']['main_protein_or_base'] * 100, 2),
        'Fine-Tuned QLoRA': round(ft_metrics['field_accuracies']['main_protein_or_base'] * 100, 2),
    },
    {
        'Métrica': 'Portion Size Accuracy (%)',
        'Base Zero-Shot': round(base_metrics['field_accuracies']['portion_size'] * 100, 2),
        'Fine-Tuned QLoRA': round(ft_metrics['field_accuracies']['portion_size'] * 100, 2),
    },
    {
        'Métrica': 'Dietary Tags Micro-F1 (%)',
        'Base Zero-Shot': round(base_metrics['dietary_tags_micro_f1'] * 100, 2),
        'Fine-Tuned QLoRA': round(ft_metrics['dietary_tags_micro_f1'] * 100, 2),
    },
    {
        'Métrica': 'Components Token F1 (%)',
        'Base Zero-Shot': round(base_metrics['components_token_f1'] * 100, 2),
        'Fine-Tuned QLoRA': round(ft_metrics['components_token_f1'] * 100, 2),
    },
    {
        'Métrica': 'Search Synonyms Token F1 (%)',
        'Base Zero-Shot': round(base_metrics['search_synonyms_token_f1'] * 100, 2),
        'Fine-Tuned QLoRA': round(ft_metrics['search_synonyms_token_f1'] * 100, 2),
    },
    {
        'Métrica': 'Food Safety Violations (Count)',
        'Base Zero-Shot': base_metrics['food_safety_violations'],
        'Fine-Tuned QLoRA': ft_metrics['food_safety_violations'],
    },
    {
        'Métrica': 'Latency P50 (ms/SKU)',
        'Base Zero-Shot': base_metrics['latency']['p50_ms'],
        'Fine-Tuned QLoRA': ft_metrics['latency']['p50_ms'],
    },
])
comparison_df['Delta'] = (comparison_df['Fine-Tuned QLoRA'] - comparison_df['Base Zero-Shot']).round(2)
print('\n=== TABELA COMPARATIVA OFICIAL (200 SKUs REAIS DE HOLDOUT) ===', flush=True)
print(comparison_df.to_string(index=False), flush=True)

In [ ]:
# [Célula 9] Exportação de Resultados e Publicação do Adaptador LoRA no Hugging Face Hub
final_payload = {
    'base_model_id': BASE_MODEL_ID,
    'dataset_id': HF_DATASET_ID,
    'adapter_repo_id': HF_ADAPTER_REPO_ID,
    'train_duration_minutes': train_duration_min,
    'final_training_loss': round(float(train_result.training_loss), 4),
    'base_zero_shot': base_metrics,
    'fine_tuned_qlora': ft_metrics,
}

out_json_path = Path('/kaggle/working/benchmark_results.json')
out_json_path.write_text(json.dumps(final_payload, indent=2, ensure_ascii=False), encoding='utf-8')
print('\n=== COPIE O BLOCO JSON ABAIXO PARA O REPOSITÓRIO LOCAL ===', flush=True)
print(json.dumps(final_payload, indent=2, ensure_ascii=False), flush=True)

adapter_local_dir = Path('/kaggle/working/qwen2.5-1.5b-ifood-catalog-qlora')
peft_model.save_pretrained(adapter_local_dir)
tokenizer.save_pretrained(adapter_local_dir)

md_table_rows = [
    '| Métrica | Base Zero-Shot | Fine-Tuned QLoRA | Delta |',
    '| :--- | :---: | :---: | :---: |',
]
for _, r in comparison_df.iterrows():
    md_table_rows.append(f"| {r['Métrica']} | {r['Base Zero-Shot']} | {r['Fine-Tuned QLoRA']} | {r['Delta']} |")

if hf_token:
    print(f'\nPublicando adaptador LoRA e Tokenizer em https://huggingface.co/{HF_ADAPTER_REPO_ID} ...', flush=True)
    peft_model.push_to_hub(HF_ADAPTER_REPO_ID, token=hf_token, private=False)
    tokenizer.push_to_hub(HF_ADAPTER_REPO_ID, token=hf_token, private=False)
    model_card_lines = [
        '---',
        'language:',
        '- pt',
        'license: mit',
        'base_model: Qwen/Qwen2.5-1.5B-Instruct',
        'datasets:',
        f'- {HF_DATASET_ID}',
        'tags:',
        '- ifood',
        '- catalog-intelligence',
        '- qlora',
        '- peft',
        '- qwen2.5',
        '---',
        '',
        f'# {HF_ADAPTER_REPO_ID}',
        '',
        'Adaptador **QLoRA (4-bit NF4, r=16, alpha=32)** treinado sobre o `Qwen/Qwen2.5-1.5B-Instruct` utilizando 1.150 SKUs reais do varejo e delivery brasileiro (`LucasFig0/brazilian-delivery-catalog-sft`).',
        '',
        '## Resultados no Holdout Set (200 SKUs Reais Inéditos)',
        '',
        *md_table_rows,
    ]
    api = HfApi(token=hf_token)
    api.upload_file(
        path_or_fileobj='\n'.join(model_card_lines).encode('utf-8'),
        path_in_repo='README.md',
        repo_id=HF_ADAPTER_REPO_ID,
        repo_type='model',
        token=hf_token,
        commit_message='Upload QLoRA benchmark results on 200 real Brazilian SKUs',
    )
    print(f'Sucesso! Modelo publicado em: https://huggingface.co/{HF_ADAPTER_REPO_ID}', flush=True)